# Overføringslæring med et ferdigtrent bildesyn

**Du skal kunne:** fryse en modell, trene et nytt utgangslag og skille dette fra
å trene fra bunnen. Vi bruker ResNet18 og CIFAR-10 igjen. Tenk på modellen som en
person som allerede kjenner mange bildemønstre og får en ny sorteringsoppgave.

Her er GPU nyttig: bildene skaleres opp til 224 × 224. Vi starter med 2000
treningsbilder, batch 16 og to epoker. Første kjøring laster ned modellvekter.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Bruk forbehandlingen som hører til vektene

`ResNet18_Weights.DEFAULT` velger Torchvisions standardvekter. `weights.transforms()`
gir tilhørende skalering, beskjæring og normalisering. Vi bruker den samme
deterministiske forbehandlingen i trening og validering her. Dataaugmentering kan
undersøkes senere. Dette er annerledes enn bare å dele pikslene på 255.

For rettferdig datasammenligning bruker vi samme seed og opprinnelige 45 000/5000-skille
som CNN-øvelsen. Startutvalget er mindre for å begrense arbeidet. Sluttesten er urørt.


In [ ]:
from torch import nn
from torch.utils.data import random_split, Subset
from torchvision import datasets, models
from kursverktoy import lag_lastere, tren, maal
vekttype = models.ResNet18_Weights.DEFAULT
forbehandling = vekttype.transforms()
alle = datasets.CIFAR10(kursmappe / "data", train=True, download=True,
                       transform=forbehandling)
trening, validering = random_split(alle, [45000, 5000],
                                  generator=torch.Generator().manual_seed(42))
trening = Subset(trening, range(2000))
validering = Subset(validering, range(500))
trening_laster, val_laster = lag_lastere(trening, validering, batch=16)
print(forbehandling)


## Frys basen og bytt siste lag

`requires_grad_(False)` slår av gradientberegning for eksisterende parametre.
Det nye `Linear`-laget lages etter frysing og er derfor lærbart. `in_features`
forteller hvor mange egenskaper ResNet sender til siste lag; vi bytter tusen
opprinnelige kategorier med ti CIFAR-kategorier.

ResNet har **BatchNorm**, som lagrer statistikk i tillegg til vekter. Å fryse
parametre alene fryser ikke denne statistikken. `frys_basen=True` gjør at
treningsfunksjonen holder basen i evalueringsmodus mens siste lag trenes.
Åpne funksjonen og finn `modell.eval()` og `modell.fc.train()`.


In [ ]:
modell = models.resnet18(weights=vekttype)
for parameter in modell.parameters():
    parameter.requires_grad_(False)
modell.fc = nn.Linear(modell.fc.in_features, 10)
modell = modell.to(enhet)
print("Lærbare:", sum(p.numel() for p in modell.parameters() if p.requires_grad))
print("Totalt:", sum(p.numel() for p in modell.parameters()))
historikk = tren(modell, trening_laster, val_laster, enhet,
                  epoker=2, lr=0.001, frys_basen=True)


## Hvilke klasser blandes sammen

En forvirringsmatrise viser fasit som rader og prediksjon som kolonner. Diagonalen
er riktig klassifisering. `bincount` teller de hundre mulige kombinasjonene;
`reshape` gjør dem til 10 × 10. Dette bruker **validering**, som vi fortsatt kan
undersøke mens vi utvikler modellen.


In [ ]:
matrise = torch.zeros(10, 10, dtype=torch.int64)
modell.eval()
with torch.no_grad():
    for bilder, fasit in val_laster:
        gjett = modell(bilder.to(enhet)).argmax(1).cpu()
        matrise += torch.bincount(10 * fasit + gjett, minlength=100).reshape(10, 10)
fig, akse = plt.subplots(figsize=(9, 7))
bilde = akse.imshow(matrise, cmap="Blues")
akse.set_xticks(range(10), alle.classes, rotation=60)
akse.set_yticks(range(10), alle.classes)
akse.set_xlabel("Gjettet")
akse.set_ylabel("Fasit")
fig.colorbar(bilde, ax=akse, label="Antall bilder")
plt.tight_layout()
plt.show()


## Gjør sammenligningen rettferdig

Ikke konkluder med at en modell er best ved å sammenligne ulike datautvalg og
treningsbudsjetter. Bruk samme trenings- og valideringsindekser som i notatbok 04,
og noter antall bilder, parametre og tid. Diskuter også at den ferdigtrente modellen
allerede har lært fra en stor ekstern bildesamling. En høyere treffandel kan dermed
komme fra mer tidligere læring, ikke bare arkitektur.

**Valgfritt forsøk:** Tren CNN-en fra bunnen på de samme 2000 bildene. Sammenlign
validering, inkludert tid og forbehandling. For finjustering kan du senere åpne
`layer4` for læring med en lavere læringsrate; det er et eget forsøk som trenger
bevisst håndtering av BatchNorm og ny optimizer.

**Kontroll:** Trener vi hele ResNet her? Nei, bare det nye utgangslaget med 5130
parametre. Vi bruker fortsatt basen for å beregne egenskaper for hvert bilde.

Kilder: [Torchvisions ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
og [PyTorch om overføringslæring](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html).
**Neste:** [06-attention-med-synlige-tall.ipynb](06-attention-med-synlige-tall.ipynb).
